# Trabajo Práctico Aprendizaje Automático 2

## Imports

In [11]:
%pip install datasets transformers torch pandas numpy scikit-learn matplotlib

  Using cached transformers-5.21.0-py3-none-any.whl.metadata (31 kB)
  Using cached torch-2.14.1-cp314-cp314-win_amd64.whl.metadata (38 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached regex-2026.9.29-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.27.3-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
%load_ext autoreload
%autoreload 2

from src.datos.carga import cargar_wikipedia
from src.datos.preprocesamiento import construir_dataset
from src.datos.analisis import resumen_etiquetas

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Tokenizador

Utilizamos el código proporcionado.

In [13]:
from transformers import BertTokenizer, BertModel
import torch

model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

def get_multilingual_token_embedding(token: str):
    """
    Devuelve el embedding (estático) para el token.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)

    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌El token '{token}' no pertenece al vocabulario de multilingual BERT.")
        return None

    embedding_vector = model.embeddings.word_embeddings.weight[token_id]

    print(f"✅Token: '{token}' | ID: {token_id}")
    print(f"Embedding shape: {embedding_vector.shape}")
    return embedding_vector

c:\Users\vicky\Downloads\TP-AA2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vicky\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\vicky\Downloads\TP-AA2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarni

## Dataset

Primero tenemos que conseguir textos para llevar a cabo la tarea. Vamos a utilizar un dataset de Wikipedia (por ahora) después lo podemos cambiar.

(Agregado giani) sumo que opensubtitles puede estar bueno para tener mas instancias de (?) y (¿). Habria que mezclar ambos

In [14]:
# aleatorio=True toma artículos al azar en vez de los primeros 100 seguidos
muestra_textos = cargar_wikipedia(n_textos=100, min_len=100, aleatorio=False)

print(f"Se descargaron {len(muestra_textos)} textos de muestra.")

# Mostramos el primer ejemplo crudo
print("\nEjemplo de texto crudo descargado:")
print(muestra_textos[0][:250] + "...")

c:\Users\vicky\Downloads\TP-AA2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vicky\.cache\huggingface\hub\datasets--wikimedia--wikipedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Se descargaron 100 textos de muestra.

Ejemplo de texto crudo descargado:
Andorra, oficialmente Principado de Andorra () es un micro-Estado soberano sin litoral ubicado en el suroeste de Europa, entre España y Francia, en el límite de la península ibérica. Se constituye en Estado independiente, de derecho, democrático y so...


Una vez que bajamos los parráfos de wikipedia, los vamos a procesar para armar el dataset que utilizaremos luego.

### Funciones de procesamiento

Están en `src/datos/preprocesamiento.py`:
- `normalizar_signos`: reemplaza o elimina los signos que no son etiquetas (`REEMPLAZOS`).
- `seccionar_palabra`: separa una palabra cruda en puntuación inicial, sub-palabras limpias y puntuación final.
- `determinar_capitalizacion`: clase de capitalización (0, 1, 2, 3) según la consigna.
- `procesar_parrafo_a_chunks`: tokeniza con BERT, asigna las etiquetas a cada sub-token y parte en chunks.
- `construir_dataset`: aplica lo anterior a todos los textos y concatena.

Reemplaza la extracción de puntuación que estaba dentro de `procesar_parrafo_a_chunks`. Antes solo se miraba el primer y el último carácter, así que en casos como `"¿Vienes?"` o `(Andorra),` se perdía la etiqueta, y los demás signos quedaban como tokens. Ahora se limpian todos los signos de los bordes, las palabras con guion se separan (`Castilla-La` → `Castilla La`) y se borran los signos internos (`EE.UU` → `EEUU`).

Aplicamos sobre la lista de parráfos bajados.

In [16]:
df_dataset_final = construir_dataset(muestra_textos, tokenizer, chunk_size=32)

print("Muestra del Dataset de Entrenamiento generado por Chunks:")
display(df_dataset_final.head(20))

Muestra del Dataset de Entrenamiento generado por Chunks:


,instancia_id,token_id,token,punt_inicial,punt_final,capitalización
0,1,10111,and,,,1
1,1,10667,##or,,,1
2,1,10288,##ra,,",",1
3,1,29059,oficialmente,,,0
4,1,109253,princ,,,1
5,1,60287,##ipa,,,1
6,1,10317,##do,,,1
7,1,10104,de,,,0
8,1,10111,and,,,1
9,1,10667,##or,,,1


Veamos cuantas instacias tenemos

In [17]:
resumen_etiquetas(df_dataset_final)

Número total de tokens (filas en CSV): 333034
Número total de instancias (chunks): 10610

Punt. inicial:
 punt_inicial
(nada)    333024
¿             10
Name: count, dtype: int64 

Punt. final:
 punt_final
(nada)    302000
,          18919
.          12104
?             11
Name: count, dtype: int64 

Capitalización (por palabra):
 capitalización
0    195964
1     35243
2       397
3      1576
Name: count, dtype: int64 

Instancias con '?': 0.10%
punt_inicial OK
punt_final OK
capitalización OK


In [18]:
# Verificar si algún chunk comienza con un sub-token
primeros_tokens = df_dataset_final.groupby("instancia_id").first()

chunks_rotos = primeros_tokens[
    primeros_tokens["token"].str.startswith("##")
]

print("Chunks que empiezan con ##:", len(chunks_rotos))

Chunks que empiezan con ##: 0


In [19]:
tamanios = df_dataset_final.groupby("instancia_id").size()

print("Tamaño mínimo:", tamanios.min())
print("Tamaño máximo:", tamanios.max())
print("Chunks con más de 32 tokens:", (tamanios > 32).sum())

Tamaño mínimo: 1
Tamaño máximo: 36
Chunks con más de 32 tokens: 1


In [20]:
# Identificar el chunk que supera los 32 tokens
ids_largos = tamanios[tamanios > 32].index

for id_chunk in ids_largos:
    chunk = df_dataset_final[
        df_dataset_final["instancia_id"] == id_chunk
    ]

    print("Instancia:", id_chunk)
    print("Cantidad de tokens:", len(chunk))
    print("Tokens:", chunk["token"].tolist())

Instancia: 1227
Cantidad de tokens: 36
Tokens: ['012', '##34', '##5', '##6', '##7', '##8', '##9', '##ab', '##c', '##de', '##f', '##ghi', '##jk', '##lm', '##no', '##p', '##q', '##rst', '##uv', '##w', '##xy', '##za', '##bc', '##de', '##f', '##ghi', '##jk', '##lm', '##no', '##p', '##q', '##rst', '##uv', '##w', '##xy', '##z']
